# 🏛️ STF Judicial Lakehouse — Backlog Temporal Survival Analysis

## 📌 Objective & Methodological Framework

This notebook implements a non-parametric **Kaplan-Meier Survival Analysis** over the judicial docket of the **Supreme Federal Court of Brazil (STF)**.

### Research Questions:
1. **Backlog Half-Life**: What is the median number of days required for a case to reach resolution (disposition/baixa)?
2. **Procedural Heterogeneity**: How do retention rates differ across legal classes (e.g. *Habeas Corpus* vs. *Recurso Extraordinário* vs. *Ação Direta de Inconstitucionalidade*)?
3. **Reporting Justice Dynamics**: What are the retention curves across the 11 judicial cabinets?
4. **Censoring Handling**: How does the model account for active cases still in tramitation (*right-censored observations*)?

$$\hat{S}(t) = \prod_{t_i \leq t} \left( 1 - \frac{d_i}{n_i} \right)$$

Where:
- $t_i$: distinct event times (disposition dates).
- $d_i$: number of cases resolved at time $t_i$.
- $n_i$: number of active cases in the risk set just prior to $t_i$.

In [ ]:
from pathlib import Path
import sys

# Ensure project root in sys.path
project_root = Path(".").resolve() if Path("data").exists() else (Path("..").resolve().parent if Path("../../data").exists() else Path("..").resolve())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import duckdb
import polars as pl
from analytics.survival import BacklogSurvivalAnalyzer

print(f"Project root resolved to: {project_root}")
print("Libraries loaded successfully.")

## 1. Connecting to the DuckDB Lakehouse Catalog
We query the curated star-schema directly.

In [ ]:
db_path = project_root / "data" / "curated" / "stf_warehouse.duckdb"
con = duckdb.connect(str(db_path), read_only=True)

df_summary = con.execute("""
    SELECT 
        situacao,
        COUNT(*) AS total_processos,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS percentual
    FROM fact_processes
    GROUP BY situacao
    ORDER BY total_processos DESC;
""").pl()

print("Status Distribution of STF Docket:")
print(df_summary)

## 2. Computing the Global Kaplan-Meier Survival Curve

In [ ]:
analyzer = BacklogSurvivalAnalyzer(db_path)
results = analyzer.compute_survival_overview()
global_km = results["global"]

print(f"Total Examined Cases: {global_km['total_cases']:,}")
print(f"Resolved Cases (Events): {global_km['events_count']:,}")
print(f"Active Cases (Right-Censored): {global_km['censored_count']:,}")
print(f"Median Backlog Duration (Half-Life): {global_km['median_days']} days\n")

print("Retention Rates at Key Milestones:")
for milestone, prob in global_km["milestone_survival"].items():
    days = milestone.replace("_days", "")
    print(f"- At {days:>4} days: {prob * 100:.1f}% of cases remain active")

## 3. Stratification by Procedural Legal Class
Comparison of survival probabilities across major actions (`HC`, `RE`, `ARE`, `ADI`, `Rcl`).

In [ ]:
classes_data = results["stratified_by_class"]
print("Median Duration and 1-Year Backlog Retention by Class:")
print(f"{'Classe':<10} | {'Median Days':<12} | {'1-Year Active %':<15} | {'Cases':<8}")
print("-" * 55)
for cls, data in classes_data.items():
    med = str(data['median_days']) if data['median_days'] else 'N/A'
    ret_1yr = f"{data['milestone_survival'].get('365_days', 0) * 100:.1f}%"
    print(f"{cls:<10} | {med:<12} | {ret_1yr:<15} | {data['total_cases']:<8}")

## 4. Stratification across Judicial Cabinets (11 Justices)

In [ ]:
judges_data = results["stratified_by_judge"]
print("Justice Workload Survival Profile:")
print(f"{'Ministro(a)':<25} | {'Median Days':<12} | {'1-Year Retention':<16} | {'Cases':<8}")
print("-" * 68)
for name, j_data in judges_data.items():
    med = str(j_data['median_days']) if j_data['median_days'] else 'N/A'
    ret = f"{j_data['one_year_retention_pct']}%"
    print(f"{name:<25} | {med:<12} | {ret:<16} | {j_data['total_cases']:<8}")

## 5. Analytical Conclusions
- **Urgent Actions** (such as *Habeas Corpus*) demonstrate rapid decay, reaching low median resolution times.
- **Abstract Review & Precedents** (*ADI*, *ADPF*, *RE*) require longer collegial deliberations, resulting in higher 1-year and 2-year backlog retention.
- **Data Integrity**: All underlying data is tracked via SHA-256 manifests and validated against the 2018–2026 temporal scope in the STF Lakehouse.